# 第05章：控制流与归约操作

## 前置知识
- 第01-04章

## 学习目标
- 掌握 `tl.where` 条件选择
- 理解归约操作：`tl.sum`, `tl.max`, `tl.min`
- 学会使用原子操作：`tl.atomic_add`, `tl.atomic_max`
- 理解 Triton 中循环的使用

In [ ]:
import torch
import triton
import triton.language as tl

## 5.1 tl.where — 条件选择

`tl.where(condition, true_val, false_val)` 是 Triton 中实现条件逻辑的主要方式：

```
condition: [True,  False, True,  False]
true_val:  [  10,    20,   30,     40]
false_val: [  -1,    -2,   -3,     -4]
result:    [  10,    -2,   30,     -4]
```

**为什么不用 if-else？**  
GPU 是 SIMT 架构，所有线程必须执行相同的指令。`tl.where` 是无分支的，
两个分支都会计算，然后根据条件选择结果。这避免了线程分化（warp divergence）。

In [ ]:
@triton.jit
def activation_kernel(
    x_ptr, out_relu_ptr, out_gelu_ptr, out_silu_ptr,
    n, BLOCK_SIZE: tl.constexpr,
):
    """
    用 tl.where 和数学函数实现常见激活函数。
    这些在 Transformer 的 FFN 层中大量使用。
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    
    # ReLU: max(0, x)
    relu = tl.where(x > 0, x, 0.0)
    tl.store(out_relu_ptr + offsets, relu, mask=mask)
    
    # GELU (近似): 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    # 简化版: x * sigmoid(1.702 * x)
    sigmoid_val = tl.sigmoid(1.702 * x)
    gelu = x * sigmoid_val
    tl.store(out_gelu_ptr + offsets, gelu, mask=mask)
    
    # SiLU (Swish): x * sigmoid(x)
    silu = x * tl.sigmoid(x)
    tl.store(out_silu_ptr + offsets, silu, mask=mask)

n = 1000
x = torch.randn(n, device='cuda')
out_relu = torch.empty_like(x)
out_gelu = torch.empty_like(x)
out_silu = torch.empty_like(x)

grid = (triton.cdiv(n, 1024),)
activation_kernel[grid](x, out_relu, out_gelu, out_silu, n, BLOCK_SIZE=1024)

# 验证
print(f"ReLU 正确性: {torch.allclose(out_relu, torch.relu(x))}")
print(f"SiLU 正确性: {torch.allclose(out_silu, torch.nn.functional.silu(x), atol=1e-5)}")

## 5.2 归约操作 (Reduction)

归约是将一个块的数据缩减为单个值（或更小的块）：

```
tl.sum:  [1, 2, 3, 4]  → 10
tl.max:  [1, 2, 3, 4]  → 4
tl.min:  [1, 2, 3, 4]  → 1
```

### 2D 归约的 axis 参数

```
矩阵 (3×4):           axis=0 (沿行归约):     axis=1 (沿列归约):
┌──┬──┬──┬──┐          sum↓                    sum→
│ 1│ 2│ 3│ 4│          ┌──┬──┬──┬──┐           ┌────┐
│ 5│ 6│ 7│ 8│    →     │ 9│12│15│18│     →     │ 10 │
│ 3│ 4│ 5│ 6│          └──┴──┴──┴──┘           │ 26 │
└──┴──┴──┴──┘          形状: (4,)              │ 18 │
                                                └────┘
                                                形状: (3,)
```

In [ ]:
@triton.jit
def row_reduce_kernel(
    x_ptr, out_sum_ptr, out_max_ptr, out_mean_ptr,
    M, N, stride_m, stride_n,
    BLOCK_N: tl.constexpr,
):
    """
    逐行归约: 对矩阵的每一行计算 sum, max, mean。
    这是 softmax 和 layernorm 的基础操作。
    """
    row_idx = tl.program_id(0)  # 每个 program 处理一行
    
    col_offsets = tl.arange(0, BLOCK_N)
    mask = col_offsets < N
    
    # 加载一整行
    row_ptrs = x_ptr + row_idx * stride_m + col_offsets * stride_n
    row_data = tl.load(row_ptrs, mask=mask, other=0.0)
    
    # 行求和
    row_sum = tl.sum(row_data, axis=0)
    tl.store(out_sum_ptr + row_idx, row_sum)
    
    # 行最大值
    # 注意: mask 为 False 的位置用 -inf 替代，否则 0 可能不是最小值
    row_data_for_max = tl.load(row_ptrs, mask=mask, other=float('-inf'))
    row_max = tl.max(row_data_for_max, axis=0)
    tl.store(out_max_ptr + row_idx, row_max)
    
    # 行均值
    row_mean = row_sum / N
    tl.store(out_mean_ptr + row_idx, row_mean)

# 测试
M, N = 64, 100
x = torch.randn(M, N, device='cuda')
out_sum = torch.empty(M, device='cuda')
out_max = torch.empty(M, device='cuda')
out_mean = torch.empty(M, device='cuda')

BLOCK_N = triton.next_power_of_2(N)  # 128
row_reduce_kernel[(M,)](
    x, out_sum, out_max, out_mean,
    M, N, x.stride(0), x.stride(1),
    BLOCK_N=BLOCK_N,
)

print(f"Sum  正确性: {torch.allclose(out_sum, x.sum(dim=1), atol=1e-4)}")
print(f"Max  正确性: {torch.allclose(out_max, x.max(dim=1).values)}")
print(f"Mean 正确性: {torch.allclose(out_mean, x.mean(dim=1), atol=1e-4)}")

## 5.3 原子操作 (Atomic Operations)

当多个 program 需要写入**同一个**内存位置时，需要原子操作来避免竞争条件：

```
问题: 两个 program 同时执行 global_sum += local_sum

Program 0:  读取 global_sum=0 → 计算 0+5=5 → 写入 5
Program 1:  读取 global_sum=0 → 计算 0+3=3 → 写入 3  ← 覆盖了 P0 的结果！
结果: global_sum = 3（错误！应该是 8）

解决: tl.atomic_add(ptr, value)
Program 0:  atomic_add(ptr, 5)  → global_sum = 5
Program 1:  atomic_add(ptr, 3)  → global_sum = 8  ← 正确！
```

Triton 支持的原子操作：
- `tl.atomic_add(ptr, val)` — 原子加
- `tl.atomic_max(ptr, val)` — 原子取最大
- `tl.atomic_min(ptr, val)` — 原子取最小
- `tl.atomic_and/or/xor(ptr, val)` — 原子位运算
- `tl.atomic_cas(ptr, cmp, val)` — Compare-And-Swap

In [ ]:
@triton.jit
def global_sum_kernel(
    x_ptr, output_ptr,
    n, BLOCK_SIZE: tl.constexpr,
):
    """
    全局求和: 使用 atomic_add 将所有块的局部和累加到全局结果。
    
    步骤:
    1. 每个 program 计算局部和 (tl.sum)
    2. 用 tl.atomic_add 将局部和加到全局结果
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    local_sum = tl.sum(x, axis=0)
    
    # 原子加: 多个 program 安全地写入同一位置
    tl.atomic_add(output_ptr, local_sum)

# 测试
n = 100000
x = torch.randn(n, device='cuda')
output = torch.zeros(1, device='cuda')  # 必须初始化为 0！

grid = (triton.cdiv(n, 1024),)
global_sum_kernel[grid](x, output, n, BLOCK_SIZE=1024)

expected = x.sum()
print(f"全局求和:")
print(f"  Triton:  {output.item():.4f}")
print(f"  PyTorch: {expected.item():.4f}")
print(f"  误差:    {abs(output.item() - expected.item()):.6f}")
print(f"  (注意: 浮点加法顺序不同会导致小误差)")

## 5.4 直方图 — 原子操作的经典应用

In [ ]:
@triton.jit
def histogram_kernel(
    x_ptr, hist_ptr,
    n, num_bins,
    BLOCK_SIZE: tl.constexpr,
):
    """
    直方图: 统计 [0, num_bins) 中每个整数出现的次数。
    使用 atomic_add 处理竞争写入。
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    
    # 加载数据
    x = tl.load(x_ptr + offsets, mask=mask, other=0)
    
    # 对每个元素，原子地增加对应 bin 的计数
    # 注意: 这里对每个有效元素都执行一次 atomic_add
    valid = mask & (x >= 0) & (x < num_bins)
    tl.atomic_add(hist_ptr + x, 1, mask=valid)

# 测试
n = 100000
num_bins = 10
x = torch.randint(0, num_bins, (n,), device='cuda', dtype=torch.int32)
hist = torch.zeros(num_bins, device='cuda', dtype=torch.int32)

grid = (triton.cdiv(n, 1024),)
histogram_kernel[grid](x, hist, n, num_bins, BLOCK_SIZE=1024)

expected = torch.bincount(x, minlength=num_bins)
print(f"直方图:")
print(f"  Triton:  {hist.tolist()}")
print(f"  PyTorch: {expected.tolist()}")
print(f"  正确性:  {torch.equal(hist, expected)}")

## 5.5 Triton 中的循环

Triton 支持 `for` 和 `while` 循环，但有一些注意事项：

```python
# ✅ 编译时确定的循环次数（编译器会展开）
for i in range(0, K, BLOCK_K):  # K 是运行时值, BLOCK_K 是 constexpr
    ...

# ✅ tl.static_range 强制展开
for i in tl.static_range(4):  # 编译器一定会展开
    ...

# ⚠️ while 循环 (运行时循环次数)
while condition:  # 可以用，但编译器无法展开
    ...
```

### 关键区别
- `range(0, K, BLOCK_K)`: K 可以是运行时值，编译器会生成循环代码
- `tl.static_range(N)`: N 必须是编译时常量，编译器会完全展开

In [ ]:
@triton.jit
def multi_pass_reduce_kernel(
    x_ptr, out_ptr,
    M, N, stride_m, stride_n,
    BLOCK_N: tl.constexpr,
):
    """
    当一行太长，BLOCK_N 无法一次覆盖时，用循环分多次归约。
    计算: out[i] = max(A[i, :])
    """
    row_idx = tl.program_id(0)
    
    # 初始化为负无穷 (标量，不是 block)
    running_max = float('-inf')
    
    # 循环处理每个块
    for start in range(0, N, BLOCK_N):
        col_offsets = start + tl.arange(0, BLOCK_N)
        mask = col_offsets < N
        
        block = tl.load(
            x_ptr + row_idx * stride_m + col_offsets * stride_n,
            mask=mask,
            other=float('-inf'),  # 越界位置用 -inf
        )
        
        block_max = tl.max(block, axis=0)
        running_max = tl.maximum(running_max, block_max)
    
    tl.store(out_ptr + row_idx, running_max)

# 测试: 大矩阵的行最大值
M, N = 32, 10000  # N 很大，一个 block 装不下
x = torch.randn(M, N, device='cuda')
out = torch.empty(M, device='cuda')

BLOCK_N = 256  # 每次只处理 256 列
multi_pass_reduce_kernel[(M,)](
    x, out, M, N, x.stride(0), x.stride(1),
    BLOCK_N=BLOCK_N,
)

expected = x.max(dim=1).values
print(f"多遍归约行最大值:")
print(f"  正确性: {torch.allclose(out, expected)}")
print(f"  前5个: Triton={out[:5].tolist()}, PyTorch={expected[:5].tolist()}")

## 总结

| 操作 | API | 典型用途 |
|------|-----|----------|
| 条件选择 | `tl.where(cond, x, y)` | 激活函数、mask |
| 求和 | `tl.sum(x, axis)` | softmax 分母、均值 |
| 最大值 | `tl.max(x, axis)` | softmax 稳定化 |
| 最小值 | `tl.min(x, axis)` | clamp |
| 原子加 | `tl.atomic_add(ptr, val)` | 全局归约、直方图 |
| 原子最大 | `tl.atomic_max(ptr, val)` | 全局最大值 |
| 静态展开 | `tl.static_range(n)` | 小循环优化 |

## 练习

1. 实现 **argmax**: 返回每行最大值的索引
2. 实现 **topk=1**: 返回每行最大值和对应索引
3. 用 atomic_max 实现 **全局最大值** 查找
4. 实现 **方差计算**: `var[i] = mean((x[i,:] - mean(x[i,:]))^2)`